# p14 — Agent trajectory drift (v1.5.0-agent-bench)

Reproduces `bench/data/p14_agent_trajectory_drift.csv`, Figure 5, and the claim gate
on a fresh Google Colab runtime.

**This workload is CPU-bound.** The harness quantizes many small (64-element) tensors;
a GPU does not help and is not used. What helps is core count, so the sweep runs with
`--jobs $(nproc)`.

Rough wall clock: ~45 min single-threaded, ~12 min on 4 vCPUs, ~6 min on 8.
The committed CSV was produced on a 4-core host, and `--jobs` is verified not to change
results (see `tests/test_agent_trajectory.py` and the determinism check in the PR).

In [ ]:
REPO = "https://github.com/ammmanism/kv-quant.git"
BRANCH = "main"  # use feat/agentic-trajectory-harness before this PR merges

!git clone --depth 1 --branch $BRANCH $REPO /content/kv-quant
%cd /content/kv-quant

In [ ]:
!pip -q install pandas matplotlib pillow

In [ ]:
import os

import torch

print("cpu_count:", os.cpu_count())
print("torch:", torch.__version__)
print("GPU irrelevant: this workload is CPU-bound small-tensor quantization")

In [ ]:
# Note the `-m`: tasks/ has no __init__.py, so `python tasks/agent_trajectory.py`
# fails with ModuleNotFoundError. Running as a module puts the repo root on sys.path.
!python -m tasks.agent_trajectory --jobs {os.cpu_count()}

In [ ]:
!python scripts/generate_phase14_artifacts.py
!python scripts/check_claims.py
!python scripts/check_figure_provenance.py

In [ ]:
import pandas as pd

df = pd.read_csv("bench/data/p14_agent_trajectory_drift.csv")
print("shape:", df.shape)

summary = df.groupby(["context_len", "config"]).agg(
    amnesia=("k_amnesia_incident", "mean"),
    v_schema=("v_schema_incident", "mean"),
    resolved=("resolved", "max"),
    kdist=("key_distortion", "mean"),
)
print(summary.round(4).to_string())

In [ ]:
from google.colab import files

for artifact in (
    "bench/data/p14_agent_trajectory_drift.csv",
    "bench/data/p14_agent_trajectory_drift.csv.manifest.json",
    "docs/figures/fig5_agent_amnesia_vs_syntax.png",
    "docs/figures/fig5_agent_amnesia_vs_syntax.png.meta.json",
):
    files.download(artifact)